# ATARRA: train the segmentation model on Colab

This notebook trains the 4-class reed segmentation model on real Sentinel-2 imagery of Lake Burullus. It runs in **two stages on purpose**:

| Stage | What it does | Cost | How often |
| --- | --- | --- | --- |
| 1. Build | Fetches imagery, tiles it, writes a store to Drive | 30-60 min, ~4 GB of ranged reads | Once |
| 2. Train | Trains from the store | minutes per run | As often as you like |

Splitting them is not tidiness. Free-tier Colab sessions cap out and disconnect when idle, so a fetch combined with a training run means a disconnect at 90% throws away both. The store is also the expensive half; training on top of it is cheap and repeatable.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`.

### The honest caveat, stated up front

The training labels come from a rule engine (weak supervision), not from field-verified annotation. So the accuracy this notebook prints measures **agreement with that rule engine** - if the rules are wrong about a pixel, a model that reproduces them faithfully is still scored correct. It is the interim number; it is not the proposal's mIoU >= 0.82 claim. The last cell in this notebook lists exactly which tiles a human should look at first, which is how that claim gets substantiated later.

In [ ]:
import sys

print('python ', sys.version.split()[0])

for name in ('numpy',):
    try:
        module = __import__(name)
        print(name.ljust(7), module.__version__)
    except Exception as exc:
        print(name.ljust(7), 'MISSING', exc)

try:
    import torch
    print('torch  ', torch.__version__)
    print('cuda   ', torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print('device ', props.name, str(round(props.total_memory / 1e9, 2)) + ' GB')
    else:
        print()
        print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.')
except Exception as exc:
    print('torch   MISSING', exc)

try:
    import psutil
    print('ram    ', str(round(psutil.virtual_memory().total / 1e9, 1)) + ' GB')
except Exception:
    pass

## Stage 1: build the tile store

Run this section once. It survives session death because it writes to Drive, and it is **resumable by design**: if the manifest is already there, the build cell does nothing rather than starting over.

The parameters below are chosen for a free-tier T4:

- **`--gsd 20`** - a full Burullus 8-band composite is 161 MB at 20 m, but 644 MB at 10 m. Twelve dates at 10 m is 7.6 GB of composites in RAM and will die on a free session. At 20 m it is 1.9 GB.
- **`--stride 128`** - half the tile size, so tiles overlap 2x. That quadruples the number of training samples for the *same* download, because the imagery is fetched once and cut afterwards. Overlapping tiles share a spatial block and therefore land in the same split, so this does not leak between train and test.
- **`--dates 12`** - spread evenly across 24 months rather than the 12 most recent. A model trained only on late summer has seen the one season where reed is easiest to separate from cropland.

In [ ]:
import os
import sys

PROJECT_DIR = '/content/atarra'
DRIVE_ROOT = '/content/drive/MyDrive/atarra'
STORE_DIR = DRIVE_ROOT + '/store_20m'
RUNS_DIR = DRIVE_ROOT + '/runs'
LOCAL_STORE = '/content/store_20m'

# This project has no git remote yet. Either set the URL here, or upload the project
# to PROJECT_DIR yourself and leave this empty.
REPO_URL = ''

IN_COLAB = 'google.colab' in sys.modules
print('in colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(RUNS_DIR, exist_ok=True)
    print()
    print('store  ->', STORE_DIR)
    print('runs   ->', RUNS_DIR)
else:
    print('not running in Colab; the build and train cells will still work')

if not REPO_URL and not os.path.isdir(PROJECT_DIR) and IN_COLAB:
    print()
    print('Set REPO_URL in this cell to clone the project, or upload it to', PROJECT_DIR)

In [ ]:
import os
import subprocess
import sys

def run(command):
    print('$', command)
    subprocess.run(command, shell=True, check=True)

if not os.path.isdir(PROJECT_DIR):
    if not REPO_URL:
        raise SystemExit('Set REPO_URL above, or place the project at ' + PROJECT_DIR)
    run('git clone --depth 1 ' + REPO_URL + ' ' + PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('working directory:', os.getcwd())

# requirements-colab.txt deliberately leaves numpy and torch alone. numpy is unpinned
# because the <2 cap in requirements.txt exists for one machine's CUDA torch build,
# and torch is absent because Colab's CUDA build is the reason to be here at all.
run(sys.executable + ' -m pip install -q -r requirements-colab.txt')

# --no-deps is load-bearing: without it, setuptools re-resolves torch and replaces the
# CUDA wheel with whichever build PyPI prefers, and the GPU is gone.
run(sys.executable + ' -m pip install -q -e . --no-deps --no-build-isolation')

run(sys.executable + ' -c "import atarra, rasterio, torch; print(atarra.__version__, rasterio.__version__, torch.__version__)"')

In [ ]:
import os
import subprocess
import sys

if os.path.exists(STORE_DIR + '/manifest.json'):
    print('a store already exists at', STORE_DIR)
    print('nothing to do. Delete that directory to rebuild it from scratch.')
else:
    command = ' '.join([
        sys.executable, '-m', 'atarra.cli', 'dataset', 'build', 'burullus',
        '--out', STORE_DIR,
        '--dates', '12',
        '--months', '24',
        '--max-cloud', '10',
        '--gsd', '20',
        '--stride', '128',
    ])
    print('$', command)
    print()
    print('This performs real reads of the satellite archive. Expect 30-60 minutes.')
    print()
    subprocess.run(command, shell=True, check=True)

In [ ]:
from atarra.datasets.store import TileStoreDataset

dataset = TileStoreDataset(STORE_DIR)
info = dataset.describe()

for key in ('area', 'gsd', 'tile_size', 'stride', 'shards', 'tiles'):
    print(key.ljust(14), info[key])

print('dates'.ljust(14), len(info['dates']))
print()

names = ['open_water', 'crops_soil', 'mixed_halophytes', 'phragmites']
total = sum(info['class_counts'])
print('class balance over all stored tiles:')
for name, count in zip(names, info['class_counts']):
    share = 100 * count / max(1, total)
    print('  ' + name.ljust(22), str(count).rjust(12), str(round(share, 2)).rjust(7) + '%')

print()
if info['class_counts'][3] == 0:
    print('WARNING: no reed pixels were stored, so the reed class cannot be learned.')
else:
    print('reed is', str(round(100 * info['class_counts'][3] / max(1, total), 2)) + '% of pixels;')
    print('inverse-frequency loss weighting is what stops the model ignoring it.')

print()
print('review share:', info['review_fraction'], '(this becomes the annotation worklist)')

## Stage 2: train

The store lives on Drive so it survives, but Drive's FUSE mount is slow for the memory-mapped, seek-heavy reads training does. So the first cell copies it to local disk once; the copy is seconds, and training then reads at full speed.

Two runs follow, and the second is not optional:

- **8-band multispectral** - visible, red-edge, NIR and SWIR.
- **RGB control arm** - blue, green, red only.

The proposal claims the multispectral stack beats RGB by 10-15% mIoU. That claim is only substantiated by running both arms on identical data with the same split and seed, which is what these two cells do. Both read the same store, so the RGB arm costs no extra download.

In [ ]:
import os
import shutil

if os.path.exists(LOCAL_STORE + '/manifest.json'):
    print('local copy already present at', LOCAL_STORE)
else:
    print('copying', STORE_DIR, '->', LOCAL_STORE)
    shutil.copytree(STORE_DIR, LOCAL_STORE)

size = 0
for root, dirs, files in os.walk(LOCAL_STORE):
    for name in files:
        size += os.path.getsize(os.path.join(root, name))

print('local store size:', str(round(size / 1e6, 1)) + ' MB')

In [ ]:
import subprocess
import sys

def train(bands, name, epochs):
    command = ' '.join([
        sys.executable, '-m', 'atarra.cli', 'train', LOCAL_STORE,
        '--bands', bands,
        '--epochs', str(epochs),
        '--batch-size', '16',
        '--workers', '2',
        '--out', RUNS_DIR,
        '--name', name,
    ])
    print('$', command)
    print()
    subprocess.run(command, shell=True, check=True)
    print()

train('8', 'unet_8band', 40)

In [ ]:
train('rgb', 'unet_rgb', 40)

In [ ]:
import json
import os

def metrics(name):
    with open(os.path.join(RUNS_DIR, name, 'metrics.json')) as handle:
        return json.load(handle)

eight = metrics('unet_8band')
rgb = metrics('unet_rgb')

print('arm'.ljust(10), 'mIoU'.rjust(8), 'reed IoU'.rjust(10), 'reed F1'.rjust(9), 'pixel acc'.rjust(11))
print('-' * 50)
for label, data in (('8-band', eight), ('rgb', rgb)):
    report = data['test_report']
    print(label.ljust(10),
          str(report['mean_iou']).rjust(8),
          str(report['phragmites_iou']).rjust(10),
          str(report['phragmites_f1']).rjust(9),
          str(report['pixel_accuracy']).rjust(11))

print()
print('per-class, 8-band:')
for entry in eight['test_report']['per_class']:
    print('  ' + entry['class_name'].ljust(24),
          'IoU', str(entry['iou']).rjust(8),
          'F1', str(entry['f1']).rjust(8),
          'support', str(entry['support_px']).rjust(10))

gain = eight['test_report']['mean_iou'] - rgb['test_report']['mean_iou']
base = rgb['test_report']['mean_iou']

print()
print('absolute gain:', str(round(gain, 4)), 'mIoU in favour of the multispectral arm')
print()

# A relative gain is only interpretable against a baseline that is actually
# measuring something. An earlier version of this cell guarded with `base > 0`, and
# since a barely-trained control arm scores ~0.0002 that guard passes and the cell
# reports 62850% -- a number that looks like a spectacular result and means nothing.
# The proposal's 10-15% claim needs both arms trained to convergence first.
MIN_INTERPRETABLE_BASELINE = 0.05
if base >= MIN_INTERPRETABLE_BASELINE:
    print('relative gain:', str(round(100 * gain / base, 1)) + '%', '(proposal target: 10-15%)')
else:
    print('relative gain NOT reported.')
    print('The RGB control arm scored', str(round(base, 4)) + ', which is too close to')
    print('zero for a ratio to be meaningful. Train both arms to convergence first.')

print()
print('CAVEAT:')
print(eight['labels']['caveat'])

## The next step that makes the number defensible

The metrics above score the model against its own teacher. To claim real detection accuracy you need a small block of hand-labelled pixels, and the tile store already knows which ones are worth your time: the rule engine recorded every pixel it was **not** confident about.

Annotating the whole queue is usually not realistic - on real Burullus imagery a majority of pixels land in it, because spectrally mixed ground genuinely is ambiguous. So the list below is **ranked**: the tiles where a human decision buys the most accuracy per minute spent come first, and each one carries its lon/lat corners so it can be opened in QGIS.

Annotate the top handful, hold them out, and report mIoU against them. That number can be defended; the one above cannot.

In [ ]:
from atarra.datasets.store import TileStoreDataset

store = TileStoreDataset(LOCAL_STORE)
everything = store.annotation_tiles()
worklist = store.annotation_tiles(limit=10)

print('tiles needing a human at all:', len(everything))
print('tiles in the store           :', len(store))
print()
print('annotate these first:')
for entry in worklist:
    print(' ', entry['key'], str(round(entry['review_fraction'] * 100, 1)) + '% review')

if worklist:
    print()
    print('corners of the top entry (lon, lat):')
    for corner in worklist[0]['corners_wgs84']:
        print(' ', corner)

In [ ]:
import os

print('artifacts under', RUNS_DIR)
print()
for root, dirs, files in os.walk(RUNS_DIR):
    for name in sorted(files):
        path = os.path.join(root, name)
        print(str(round(os.path.getsize(path) / 1e6, 2)).rjust(8) + ' MB  ' + path)

print()
print('Each run directory holds best.pt (weights plus the normalisation buffers, so')
print('inference cannot use different statistics from training), metrics.json, and')
print('history.json. They are on Drive, so they survive this session.')